In [ ]:
import pandas as pd
import numpy as np


In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
df = pd.read_csv('/content/data (1).xlsx - Sheet1.csv')
df.head()

,Unnamed: 0,Order No.,Time(UTC+8),Pair,Type,Leverage,DealPrice,Amount,Fee,Realized PNL,Order Type,AvgPrice
0,15804430,1716653171923955712,2023-10-24 11:09:45,LTC-USDT,Open Long,20X,71.620,21,-0.0107,0.0,Autonomous Orders,NaN
1,15804430,1716653174780276736,2023-10-24 11:09:45,LINK-USDT,Open Long,20X,10.444,20,-0.0099,0.0,Autonomous Orders,NaN
2,15804430,1716653178873917440,2023-10-24 11:09:46,COMP-USDT,Open Long,20X,47.990,20,-0.0098,0.0,Autonomous Orders,NaN
3,15804430,1716653182887866368,2023-10-24 11:09:47,SOL-USDT,Open Long,20X,32.225,32,-0.0161,0.0,Autonomous Orders,NaN
4,15804430,1716653185379282944,2023-10-24 11:09:48,RUNE-USDT,Open Long,20X,2.231,20,-0.0100,0.0,Autonomous Orders,NaN


**Position status**




In [ ]:

dfoa = df[['Pair','Type','Amount']] #df order amount



In [ ]:
#Unclosed order amount
open_long = dfoa[dfoa['Type'] == 'Open Long'].groupby('Pair')['Amount'].sum()
close_long = dfoa[dfoa['Type'] == 'Close Long'].groupby('Pair')['Amount'].sum()
longhold = open_long.subtract(close_long, fill_value=0)

open_short = dfoa[dfoa['Type'] == 'Open Short'].groupby('Pair')['Amount'].sum()
close_short = dfoa[dfoa['Type'] == 'Close Short'].groupby('Pair')['Amount'].sum()
shorthold = open_short.subtract(close_short, fill_value=0)

unclosed_order = pd.DataFrame(list(longhold.items()), columns=['Pair', 'Long Difference'])
unclosed_order['Short Difference'] = unclosed_order['Pair'].map(shorthold).fillna(0)
print(unclosed_order)


           Pair  Long Difference  Short Difference
0      ADA-USDT              196               0.0
1      BTC-USDT              299               0.0
2     COMP-USDT                1               0.0
3     DOGE-USDT               -4               0.0
4      DOT-USDT                0               0.0
5      ETC-USDT                0               1.0
6      ETH-USDT              197               0.0
7      FIL-USDT                0               1.0
8      INJ-USDT                0               0.0
9     LINK-USDT                1               0.0
10    LOOM-USDT               -6               0.0
11     LPT-USDT                0               0.0
12     LTC-USDT                0               0.0
13   MATIC-USDT                0               0.0
14    MINA-USDT                0              -1.0
15  PEOPLE-USDT                0               0.0
16    RUNE-USDT                0               1.0
17    SHIB-USDT                0               0.0
18     SNX-USDT                

**PnL (after removing large unclosed order)**

In [ ]:
#Order value by pair
df['Order_value'] = df['DealPrice'] * df['Amount']
dfov = df[['Pair','Type','Order_value']] #df order value


open_long_value = dfov[dfov['Type'] == 'Open Long'].groupby('Pair')['Order_value'].sum()
close_long_value = dfov[dfov['Type'] == 'Close Long'].groupby('Pair')['Order_value'].sum()
longhold_value = close_long_value.subtract(open_long_value, fill_value=0)

open_short_value = dfov[dfov['Type'] == 'Open Short'].groupby('Pair')['Order_value'].sum()
close_short_value = dfov[dfov['Type'] == 'Close Short'].groupby('Pair')['Order_value'].sum()
shorthold_value = open_short_value.subtract(close_short_value, fill_value=0)

order_value = pd.DataFrame(list(longhold_value.items()), columns=['Pair', 'Long PnL'])
order_value['Short PnL'] = order_value['Pair'].map(shorthold_value).fillna(0)
order_value = order_value.drop([0,1,6])
long_pnl = order_value['Long PnL'].sum()
short_pnl = order_value['Short PnL'].sum()
Pnl = long_pnl + short_pnl
AveragePnL = Pnl/len(dfov)
print(long_pnl)
print(short_pnl)
print(Pnl)
print(AveragePnL)
print(order_value)


-426.1736802560008
114.87453985599797
-311.29914040000284
-0.04805482253782075
           Pair      Long PnL     Short PnL
2     COMP-USDT -1.675700e+02 -5.520000e+00
3     DOGE-USDT  5.114800e-01 -3.400000e-03
4      DOT-USDT -3.420000e+00  7.400000e-02
5      ETC-USDT -2.310000e+01  3.837400e+01
7      FIL-USDT -1.288000e+00  5.071000e+00
8      INJ-USDT  2.703000e+00 -7.049000e+00
9     LINK-USDT -2.509200e+01  7.249000e+00
10    LOOM-USDT  1.284600e+00  0.000000e+00
11     LPT-USDT -5.160000e+00  5.800000e-01
12     LTC-USDT -8.029000e+01  7.056000e+01
13   MATIC-USDT  1.280000e-02 -6.400000e-03
14    MINA-USDT  7.560000e-02 -6.979000e-01
15  PEOPLE-USDT -4.560000e-03 -2.360000e-03
16    RUNE-USDT -4.738000e+00  2.408000e+00
17    SHIB-USDT -2.560000e-07 -1.440000e-07
18     SNX-USDT -1.888000e+00  5.240000e-01
19     SOL-USDT -1.178080e+02  3.776000e+00
20     WLD-USDT -4.172000e-01 -2.936000e-01
21     XRP-USDT  1.460000e-02 -1.688000e-01


**Fee & Return**

In [132]:
df_fee = df.groupby('Pair')['Fee'].sum()
order_value['Fee'] = order_value['Pair'].map(df_fee).fillna(0)
order_value['Return'] = order_value['Long PnL'] + order_value['Short PnL'] + order_value['Fee']
order_value.sort_values(by='Return', ascending=False, inplace=True)
print(order_value)
print(f'Return of port is {order_value["Return"].sum()}')

           Pair      Long PnL     Short PnL     Fee     Return
5      ETC-USDT -2.310000e+01  3.837400e+01 -1.1731   14.10090
7      FIL-USDT -1.288000e+00  5.071000e+00 -1.0444    2.73860
10    LOOM-USDT  1.284600e+00  0.000000e+00 -0.0631    1.22150
13   MATIC-USDT  1.280000e-02 -6.400000e-03 -0.0740   -0.06760
3     DOGE-USDT  5.114800e-01 -3.400000e-03 -0.6458   -0.13772
15  PEOPLE-USDT -4.560000e-03 -2.360000e-03 -0.5799   -0.58682
17    SHIB-USDT -2.560000e-07 -1.440000e-07 -0.6360   -0.63600
14    MINA-USDT  7.560000e-02 -6.979000e-01 -0.1138   -0.73610
21     XRP-USDT  1.460000e-02 -1.688000e-01 -0.7289   -0.88310
18     SNX-USDT -1.888000e+00  5.240000e-01 -0.9470   -2.31100
20     WLD-USDT -4.172000e-01 -2.936000e-01 -2.1480   -2.85880
16    RUNE-USDT -4.738000e+00  2.408000e+00 -1.0501   -3.38010
4      DOT-USDT -3.420000e+00  7.400000e-02 -0.7789   -4.12490
11     LPT-USDT -5.160000e+00  5.800000e-01 -0.8269   -5.40690
8      INJ-USDT  2.703000e+00 -7.049000e+00 -2.4958   -